# Trabalho 3 (DCA3702): Projeto Final RideSmart

Disciplina: Algoritmos e Estrutura de Dados II (DCA3702), UFRN.  
Integrantes: Lucas Augusto Spinola Pinto, João Pedro Araújo Ramalho, Kiev Luiz Freitas Guedes, Maria Eduarda Silva da Costa.

## Problema
Dado um ponto de origem A, um destino B e uma distância máxima X que o usuário aceita caminhar, escolher o melhor ponto de embarque P. A rota completa fica: A para P caminhando, P para B de carro.

Cenário escolhido: Setor de Aulas IV (UFRN, Lagoa Nova) até o Comando do 3o Distrito Naval (Santos Reis, Natal/RN).

## Pipeline
1. Baixar rede viária de Natal (drive e walk) via OSMnx.
2. Adicionar pesos: length, travel_time (sem trânsito) e travel_time_synth (com trânsito).
3. Implementar 4 algoritmos: Dijkstra simples, Dijkstra com heap, A* com heurística de Haversine, Dijkstra Bidirecional.
4. Comparar custo, nós expandidos e tempo de execução.
5. Rodar o RideSmart: sweep em X com e sem trânsito.
6. Visualizar rotas e gerar dashboard.

## 1. Setup

In [1]:
# !pip install -r requirements.txt

import os, sys, json
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
print('Working dir:', os.getcwd())

import pandas as pd
from src import graph, traffic, algorithms, ridesmart, metrics, visualization, dashboard

Working dir: C:\Users\Lucas Spinola.DESKTOP-NGIDNEL\Desktop\T2-AED2\T3


## 2. Etapa 1: Construção do grafo

Baixamos dois grafos da região via OSMnx: G_drive (vias para carro) e G_walk (caminhada). Eles cobrem um círculo de 10 km em torno do ponto médio entre A e B.

In [2]:
G_drive, G_walk = graph.baixar_grafos(cache_dir='data/cache')
print(f'G_drive: {G_drive.number_of_nodes()} nos, {G_drive.number_of_edges()} arestas')
print(f'G_walk:  {G_walk.number_of_nodes()} nos, {G_walk.number_of_edges()} arestas')

  [graph] baixando rede 'drive' em (-5.806285000000001, -35.198615000000004), raio=10000 m


  [graph] baixando rede 'walk' em (-5.806285000000001, -35.198615000000004), raio=10000 m


G_drive: 21660 nos, 56285 arestas
G_walk:  30935 nos, 90174 arestas


## 3. Etapa 2: Pesos das arestas

Cada aresta de G_drive recebe length (metros), speed_kph e travel_time (segundos). Quando o maxspeed está ausente, usamos defaults por tipo de via (residential=30 km/h, primary=60 km/h, trunk=70 km/h, etc.).

In [3]:
G_drive = graph.adicionar_pesos(G_drive)
G_walk = graph.adicionar_length_walk(G_walk)
# Inspeciona algumas arestas
for i, (u, v, data) in enumerate(G_drive.edges(data=True)):
    if i >= 3: break
    print(f'  aresta {u}->{v}: length={data["length"]:.1f}m, '
          f'highway={data.get("highway_norm")}, speed={data["speed_kph"]:.1f}km/h, '
          f'travel_time={data["travel_time"]:.1f}s')

  aresta 243207469->7051202996: length=6.5m, highway=primary, speed=70.0km/h, travel_time=0.3s
  aresta 243207479->7051202993: length=7.4m, highway=primary, speed=70.0km/h, travel_time=0.4s
  aresta 243207483->2247317589: length=9.9m, highway=primary, speed=70.0km/h, travel_time=0.5s


## 4. Etapa 2.1: Trânsito sintético

Aplicamos um fator multiplicativo no travel_time de cada aresta, correlacionado com o tipo da via:
- motorway, trunk, primary: fator entre 1,5 e 2,5 (avenidas mais congestionadas).
- secondary, tertiary: fator entre 1,2 e 1,8.
- residential, unclassified: fator entre 1,0 e 1,3.

Seed fixa para garantir reproducibilidade.

In [4]:
G_drive = traffic.aplicar_transito_sintetico(G_drive, seed=42)
resumo_t = traffic.resumo_transito(G_drive)
print(f'Fator medio geral: {resumo_t["media_geral"]:.3f}')
print('Top tipos por fator:')
for tipo, fator in list(resumo_t['media_por_tipo'].items())[:6]:
    print(f'  {tipo:18s} fator medio = {fator:.3f}')

Fator medio geral: 1.242
Top tipos por fator:
  trunk_link         fator medio = 2.018
  primary            fator medio = 2.002
  trunk              fator medio = 1.979
  primary_link       fator medio = 1.820
  secondary_link     fator medio = 1.514
  secondary          fator medio = 1.496


## 5. Localizacao dos nós para A e B

Coordenadas: A = Setor de Aulas IV (UFRN), B = 3o Distrito Naval. Usamos `osmnx.distance.nearest_nodes` para achar os nós mais próximos em cada grafo.

In [5]:
nos = graph.encontrar_nos(G_drive, G_walk)
print(f'no_a_walk  = {nos["no_a_walk"]}')
print(f'no_a_drive = {nos["no_a_drive"]}')
print(f'no_b_drive = {nos["no_b_drive"]}')
print(f'A = {nos["coord_a"]}')
print(f'B = {nos["coord_b"]}')

no_a_walk  = 530586261
no_a_drive = 784697934
no_b_drive = 8633307716
A = (-5.84245, -35.19975)
B = (-5.77012, -35.19748)


## 6. Etapa 3: Comparação dos 4 algoritmos

Rodamos cada um dos algoritmos no mesmo par (A -> B) usando o peso travel_time e tabulamos:
- Custo encontrado (deve ser idêntico, pois os 4 são ótimos).
- Número de nós expandidos.
- Tempo de execução em ms.

In [6]:
df_alg = metrics.comparar_algoritmos(G_drive, nos['no_a_drive'], nos['no_b_drive'], weight='travel_time')
df_alg

,algoritmo,custo_s,nos_expandidos,elapsed_ms,tamanho_caminho
0,dijkstra_simples,714.367064,11885,14002.5789,113
1,dijkstra_heap,714.367064,11886,37.2367,113
2,a_star,714.367064,4799,51.2305,113
3,dijkstra_bidirecional,714.367064,5079,19.2171,113


In [7]:
metrics.salvar_csv(df_alg, 'results/comparacao_algoritmos.csv')
print('comparacao_algoritmos.csv salvo.')

comparacao_algoritmos.csv salvo.


## 7. Etapa 4: RideSmart com sweep em X

Para cada X em [0, 200, 500, 800, 1000] metros, escolhemos o melhor P (que minimiza o tempo total) tanto no cenário sem trânsito quanto no cenário com trânsito sintético. Usamos Dijkstra com heap como algoritmo de caminho mínimo nesta etapa (rápido e ótimo).

In [8]:
valores_x = [0, 200, 500, 800, 1000]
sw_sem = ridesmart.sweep_x(G_drive, G_walk, nos['no_a_walk'], nos['no_b_drive'], valores_x, weight='travel_time', algoritmo=algorithms.dijkstra_heap)
sw_com = ridesmart.sweep_x(G_drive, G_walk, nos['no_a_walk'], nos['no_b_drive'], valores_x, weight='travel_time_synth', algoritmo=algorithms.dijkstra_heap)
df_sem = metrics.tabela_sweep_x(sw_sem, cenario='sem_transito')
df_com = metrics.tabela_sweep_x(sw_com, cenario='com_transito')
df_sweep = pd.concat([df_sem, df_com], ignore_index=True)
df_sweep = metrics.ganho_caminhada(df_sweep)
df_sweep

,cenario,x_metros,d_walk_m,t_walk_s,d_drive_m,t_drive_s,d_total_m,t_total_s,ganho_pct
0,sem_transito,0.0,0.0,0.0,10614.525572,714.367064,10614.525572,714.367064,0.0
1,sem_transito,200.0,0.0,0.0,10614.525572,714.367064,10614.525572,714.367064,0.0
2,sem_transito,500.0,0.0,0.0,10614.525572,714.367064,10614.525572,714.367064,0.0
3,sem_transito,800.0,0.0,0.0,10614.525572,714.367064,10614.525572,714.367064,0.0
4,sem_transito,1000.0,0.0,0.0,10614.525572,714.367064,10614.525572,714.367064,0.0
5,com_transito,0.0,0.0,0.0,10584.414949,1211.817882,10584.414949,1211.817882,0.0
6,com_transito,200.0,0.0,0.0,10584.414949,1211.817882,10584.414949,1211.817882,0.0
7,com_transito,500.0,0.0,0.0,10584.414949,1211.817882,10584.414949,1211.817882,0.0
8,com_transito,800.0,0.0,0.0,10584.414949,1211.817882,10584.414949,1211.817882,0.0
9,com_transito,1000.0,0.0,0.0,10584.414949,1211.817882,10584.414949,1211.817882,0.0


In [9]:
metrics.salvar_csv(df_sweep, 'results/comparacao_X.csv')
print('comparacao_X.csv salvo.')

comparacao_X.csv salvo.


## 8. Etapa 5: Visualizações

Geramos 6 imagens estáticas e um mapa folium interativo com as rotas escolhidas.

In [10]:
imagens_paths = {
    'rede':       'imagens/01_rede_estudo.png',
    'rotas':      'imagens/02_rotas_comparadas.png',
    'candidatos': 'imagens/03_candidatos_P.png',
    'tempo_x':    'imagens/04_tempo_vs_X.png',
    'nodes':      'imagens/05_nodes_expanded.png',
    'runtime':    'imagens/06_runtime_algoritmos.png',
}
visualization.plot_rede_estudo(G_drive, graph.COORD_A, graph.COORD_B, imagens_paths['rede'])

res_d = algorithms.dijkstra_heap(G_drive, nos['no_a_drive'], nos['no_b_drive'], weight='length')
res_t = algorithms.dijkstra_heap(G_drive, nos['no_a_drive'], nos['no_b_drive'], weight='travel_time')
res_c = algorithms.dijkstra_heap(G_drive, nos['no_a_drive'], nos['no_b_drive'], weight='travel_time_synth')
rotas_comparadas = {
    'Menor distância': res_d['path'],
    'Menor tempo (sem trânsito)': res_t['path'],
    'Menor tempo (com trânsito)': res_c['path'],
}
visualization.plot_rotas(G_drive, rotas_comparadas, graph.COORD_A, graph.COORD_B, imagens_paths['rotas'])

cand_500 = ridesmart.candidatos_p(G_walk, nos['no_a_walk'], 500)
visualization.plot_candidatos(G_walk, nos['no_a_walk'], cand_500, 500, imagens_paths['candidatos'])

visualization.plot_tempo_vs_x(df_sweep, imagens_paths['tempo_x'])
visualization.plot_nodes_expanded(df_alg, imagens_paths['nodes'])
visualization.plot_runtime(df_alg, imagens_paths['runtime'])
print('6 imagens geradas em imagens/.')

6 imagens geradas em imagens/.


In [11]:
rotas_para_mapa = {}
for r in sw_com:
    if 'erro' in r: continue
    x = r['x_metros']
    rotas_para_mapa[f'X = {int(x)} m (com transito)'] = r.get('caminho_drive', [])
visualization.mapa_interativo(G_drive, graph.COORD_A, graph.COORD_B, rotas_para_mapa, 'results/rota_interativa.html')
print('rota_interativa.html gerado.')

rota_interativa.html gerado.


## 9. Exportações finais

In [12]:
graph.salvar_graphml(G_drive, 'results/grafo.graphml')
resumo = metrics.resumo_geral(df_alg, df_sem, df_com, resumo_t)
metrics.salvar_metricas_json(resumo, 'results/metricas.json')
dashboard.gerar(df_alg=df_alg, df_sweep=df_sweep, resumo=resumo, imagens=imagens_paths, iframe_mapa='rota_interativa.html', caminho_saida='results/index.html')
print('grafo.graphml + metricas.json + index.html salvos.')

grafo.graphml + metricas.json + index.html salvos.


## 10. Discussao e conclusões

Achados principais:
- Os 4 algoritmos chegam ao mesmo custo ótimo (custo idêntico, em segundos). Confirma que estao todos corretos.
- O Dijkstra simples (sem heap) e muito mais lento que as outras versoes por causa do O(V^2): para esse grafo (~21 mil nos), ele leva varios segundos para um par. As versoes com heap, A* e Bidirecional rodam em menos de 60 ms.
- O A* expande aproximadamente 40% dos nós do Dijkstra com heap. Bem próximo do Bidirecional (43%). O ganho do A* vem da heurística geográfica admissível (Haversine convertido em segundos pela vmax do grafo).
- O Dijkstra Bidirecional foi o mais rápido em tempo de execução (cerca de 17 ms), batendo até o Heap puro porque expande menos nos.
- O trânsito sintético aumenta o tempo total da viagem em cerca de 70% (de 11,9 min para 20,2 min). E coerente com fatores entre 1,5 e 2,5 nas avenidas principais.
- O sweep em X mostra um resultado interessante: para esse cenário específico, caminhar não melhora o tempo total da rota. A explicação e que o no de A em G_drive (Setor de Aulas IV) já está sobre a BR-101 / Av. Senador Salgado Filho, que e a via mais rápida da região. Qualquer P alternativo dentro de 1 km cai em vias secundárias mais lentas, e o tempo de caminhada não compensa.

Limitacoes:
- Trânsito sintético simplifica muita coisa: trafego real depende de hora, dia da semana, eventos.
- A topologia do cenário UFRN -> Marinha favorece a rota direta pela BR-101. Em outros cenários (origem em uma rua interna de bairro), caminhar deve fazer diferença.
- max_candidatos=20 limita o número de P avaliados por X. Aumentar isso aumenta o custo computacional.

Conclusões:
- A escolha de algoritmo importa muito em redes viárias grandes. A* e Bidirecional são escolhas pragmaticas, com ganho mensuravel sobre o Dijkstra padrão.
- O RideSmart e uma ideia simples mas exige modelagem cuidadosa do trade-off entre caminhar e dirigir. O resultado pode ser 'não caminhar e a melhor opção', dependendo da topologia da região de origem.